In [ ]:
!git clone https://github.com/Mohamedmagdy21/sentiment_analyzer.git

%cd sentiment_analyzer

!pip install -r requirements.txt
# Remove torchvision (causes nms operator crash with Kaggle's torch)
!pip uninstall torchvision -y

In [ ]:
import os
os.makedirs("Data/raw/archive(1)", exist_ok=True)
!cp /kaggle/input/amazon-fine-food-reviews/Reviews.csv \
    "Data/raw/archive(1)/Reviews.csv"
print("Copied raw Amazon data from Kaggle input.")

In [ ]:
!python -m preprocessing.preprocess \
    dataset=amazon \
    preprocessing=amazon_preprocessor \
    model=amazon_roberta \
    hydra.run.dir=.

In [ ]:
!MLFLOW_TRACKING_URI=https://b3e51e750c3a8541-154-237-28-32.serveousercontent.com \
  CUDA_VISIBLE_DEVICES="" HYDRA_FULL_ERROR=1 python -m training.train \
    dataset=amazon \
    model=amazon_roberta \
    hydra.run.dir=.

In [ ]:
import shutil, os
shutil.rmtree("Data", ignore_errors=True)
shutil.rmtree(".git", ignore_errors=True)
shutil.rmtree("artifacts/checkpoints", ignore_errors=True)
shutil.rmtree("artifacts/logs", ignore_errors=True)
print("Cleaned up large files from output.")

In [ ]:
import os, json, subprocess
model_dir = "artifacts/models/amazon"
os.makedirs(model_dir, exist_ok=True)
meta = {"id": "mohamedmagdyw/amazon-roberta", "title": "Amazon RoBERTa Sentiment Model"}
with open(os.path.join(model_dir, "dataset-metadata.json"), "w") as f:
    json.dump(meta, f)
r = subprocess.run(["kaggle", "datasets", "version", "-p", model_dir, "-m", f"v33 - {os.environ.get('KAGGLE_KERNEL_VERSION','?')}"], capture_output=True, text=True)
print(r.stdout[-500:] if len(r.stdout)>500 else r.stdout)
if r.returncode != 0:
    r2 = subprocess.run(["kaggle", "datasets", "create", "-p", model_dir], capture_output=True, text=True)
    print(r2.stdout[-500:] if len(r2.stdout)>500 else r2.stdout)
    print(r2.stderr[:500] if r2.stderr else "")